# Multi-output regression

This is **not** a deeper version of [multiple regression](linear-regression.ipynb) —
it's a different problem *shape*:

- **Multiple** regression: many predictors → **one** target.
- **Multi-output** regression: many predictors → **many** targets *at once*.

Concretely: predict both a house's **price** and its **days-on-market** from the
same features, together. The naming similarity invites confusion, so hold onto
that distinction before any code.

In [ ]:
:dep ndarray = { version = "0.15" }
:dep smartcore = { version = "0.3" }
use ndarray::{Array2};

// Two features; a shared latent factor `z` influences BOTH targets but is not a
// feature — so the two targets share structure the model can't see.
let n = 80usize;
let x1: Vec<f64> = (0..n).map(|i| (i as f64 * 0.11).sin() * 4.0 + i as f64 * 0.1).collect();
let x2: Vec<f64> = (0..n).map(|i| (i as f64 * 0.19).cos() * 3.0 + 6.0).collect();
let z:  Vec<f64> = (0..n).map(|i| (i as f64 * 0.37).sin() * 2.0).collect();      // latent, not given to the model
let y1: Vec<f64> = (0..n).map(|i| 3.0 * x1[i] + 2.0 * x2[i] + z[i] + (((i*7)%5) as f64 - 2.0) * 0.2).collect();
let y2: Vec<f64> = (0..n).map(|i| -1.0 * x1[i] + 4.0 * x2[i] + z[i] + (((i*13)%5) as f64 - 2.0) * 0.2).collect();

// Gauss-Jordan inverse (small matrices only) — no LAPACK backend needed.
fn inverse(m: &Array2<f64>) -> Array2<f64> {
    let n = m.nrows();
    let (mut a, mut inv) = (m.clone(), Array2::<f64>::eye(n));
    for col in 0..n {
        let piv = a[[col, col]];
        for j in 0..n { a[[col, j]] /= piv; inv[[col, j]] /= piv; }
        for row in 0..n { if row != col { let f = a[[row, col]]; for j in 0..n { a[[row, j]] -= f * a[[col, j]]; inv[[row, j]] -= f * inv[[col, j]]; } } }
    }
    inv
}
fn r2(actual: &[f64], pred: &[f64]) -> f64 {
    let m = actual.iter().sum::<f64>() / actual.len() as f64;
    let ss_tot: f64 = actual.iter().map(|a| (a - m).powi(2)).sum();
    let ss_res: f64 = actual.iter().zip(pred).map(|(a, p)| (a - p).powi(2)).sum();
    1.0 - ss_res / ss_tot
}
fn rmse(actual: &[f64], pred: &[f64]) -> f64 {
    (actual.iter().zip(pred).map(|(a, p)| (a - p).powi(2)).sum::<f64>() / actual.len() as f64).sqrt()
}
println!("{} samples, 2 features, 2 targets (price-like y1, days-like y2)", n);

```{note}
**Crate support.** `smartcore` / `linfa` linear regression is **single-target** —
you can't pass a multi-column target matrix into their `.fit()`. So multi-output
needs either one model per target (below) or a hand-rolled solution.
```

## Baseline: one independent model per target

The cheapest approach — fit `smartcore`'s single-target regression once per
target. Simple, but it **can't exploit correlation between the targets**:

In [ ]:
{
    use smartcore::linalg::basic::matrix::DenseMatrix;
    use smartcore::linear::linear_regression::LinearRegression;
    let x = DenseMatrix::new(n, 2, (0..n).flat_map(|i| [x1[i], x2[i]]).collect(), false);
    for (name, y) in [("y1", &y1), ("y2", &y2)] {
        let lr = LinearRegression::fit(&x, y, Default::default()).unwrap();
        let pred = lr.predict(&x).unwrap();
        println!("independent {}: R^2 = {:.3}, RMSE = {:.3}", name, r2(y, &pred), rmse(y, &pred));
    }
}

## True multivariate least squares, by hand

The closed-form OLS solution generalizes directly: with the targets stacked into
an `n×k` matrix `Y`, the coefficients become a matrix `B = (XᵀX)⁻¹XᵀY`. Same
matrix machinery as the [gradient-descent chapter](../05b-optimization/gradient-descent-variants.ipynb),
generalized to multiple output columns:

In [ ]:
let residuals: (Vec<f64>, Vec<f64>) = {
    // X with a leading bias column (n x 3); Y is n x 2.
    let mut x = Array2::<f64>::ones((n, 3));
    let mut ym = Array2::<f64>::zeros((n, 2));
    for i in 0..n { x[[i, 1]] = x1[i]; x[[i, 2]] = x2[i]; ym[[i, 0]] = y1[i]; ym[[i, 1]] = y2[i]; }
    let xt = x.t().to_owned();
    let b = inverse(&xt.dot(&x)).dot(&xt.dot(&ym));   // 3 x 2 coefficient matrix
    let pred = x.dot(&b);                             // n x 2 predictions
    let p1: Vec<f64> = (0..n).map(|i| pred[[i, 0]]).collect();
    let p2: Vec<f64> = (0..n).map(|i| pred[[i, 1]]).collect();
    println!("multivariate y1: R^2 = {:.3}, RMSE = {:.3}", r2(&y1, &p1), rmse(&y1, &p1));
    println!("multivariate y2: R^2 = {:.3}, RMSE = {:.3}", r2(&y2, &p2), rmse(&y2, &p2));
    let agg = (rmse(&y1, &p1) + rmse(&y2, &p2)) / 2.0;
    println!("aggregate (mean per-output RMSE) = {:.3}", agg);
    (y1.iter().zip(&p1).map(|(a, p)| a - p).collect(), y2.iter().zip(&p2).map(|(a, p)| a - p).collect())
};

The per-output numbers match the independent baseline closely — for a *linear*
multivariate fit they're mathematically equivalent per column. The single
**aggregate** metric (mean per-output RMSE) is convenient for ranking models,
while the per-output numbers stay more informative — and you should compare on
both, since a model can win on one target and lose on another.

## The multi-output-specific diagnostic: residual correlation

Do the errors on `y1` correlate with the errors on `y2`? If so, the targets share
structure the model isn't capturing (here, the latent `z`). This diagnostic has
**no single-output equivalent**:

In [ ]:
{
    let (r1, r2v) = &residuals;
    let mean = |v: &[f64]| v.iter().sum::<f64>() / v.len() as f64;
    let (m1, m2) = (mean(r1), mean(r2v));
    let cov: f64 = r1.iter().zip(r2v).map(|(a, b)| (a - m1) * (b - m2)).sum();
    let s1: f64 = r1.iter().map(|a| (a - m1).powi(2)).sum::<f64>().sqrt();
    let s2: f64 = r2v.iter().map(|b| (b - m2).powi(2)).sum::<f64>().sqrt();
    println!("correlation between y1 and y2 residuals = {:.3}", cov / (s1 * s2));
    println!("(strongly positive -> the targets share hidden structure `z` the features don't capture)");
}

## When multi-output actually helps

Be honest about it: if the targets are genuinely **unrelated**, independent
per-output models are simpler and just as good. Multi-output modelling pays off
specifically when the targets **share structure** a joint model can exploit — and
the residual-correlation plot above is exactly how you check that on real data,
rather than assuming multi-output is always better.

```{note}
Both the multivariate OLS and the `(XᵀX)⁻¹` inverse here are **hand-rolled** with
`ndarray` — `smartcore`/`linfa` are single-target, and we avoid an
`ndarray-linalg` LAPACK dependency (see the [crate reference](../appendix/crate-reference.md)).
```

Next: [logistic regression](logistic-regression.ipynb) — regression's method,
adapted to predict categories.